In [5]:
# mapas_supervisores.py
import os
import pandas as pd
import folium
from folium.plugins import MarkerCluster

# --- CONFIG ---
dados_path = r"C:\Users\tbekho01.ATKEARNEY_AD\Kearney\Gran Coffee - Otimização de despesas operacionais - Project Management\5. Working Folder\24. Handover\250826_Otimização de Abastecedores_vEnviado\Dados Raw\Rotas para Análise\Rotas SP - Kearney - Handover.xlsx"
sheet_name = 0  # ou o nome da aba: 'Sheet1'
output_folder = r"C:\Users\tbekho01.ATKEARNEY_AD\Kearney\Gran Coffee - Otimização de despesas operacionais - Project Management\5. Working Folder\24. Handover\250826_Otimização de Abastecedores_vEnviado\Dados Intermediários"  # ajuste onde quer salvar
supervisores = ["ALAN FONTES", "ANDRÉ"]
lat_col = "Latitude"
lon_col = "Longitude"
label_col = "CLIENTE AJUSTADO ÚNICO"
supervisor_col = "SUPERVISOR"
# ---------------

os.makedirs(output_folder, exist_ok=True)

# Dependências: pip install pandas folium openpyxl
print("Lendo a planilha...")
df = pd.read_excel(dados_path, sheet_name=sheet_name, engine="openpyxl", skiprows=1)

# limpeza mínima: remover linhas sem coordenadas
df = df.copy()


Lendo a planilha...


In [6]:
df

,FILIAL,CONTRATO,PARCEIRO,PATRIMÔNIO,"PATRIMONIO sem ""0""",MODELO,CAPACIDADE EM DOSES,TIPO DE MAQUINA,PDV,CLIENTE,...,HOSPITAL / LABORATÓRIO,Dia de Inventário,Seg,Ter,Qua,Qui,Sex,Sáb,Dom,Frequência Atual
0,SP,24347.0,60146.0,4862,4862,MAQ SUPER-AUTOMATICA SAECO INTELIA,100,BEBIDAS QUENTES,COPA TÉRREO,11° CARTÓRIO,...,NaN,QUINTA-FEIRA,X,X,X,X,X,NaN,NaN,5
1,SP,24347.0,60146.0,082001,82001,MAQ VENDING BQ NECTA BRIO 250,250,BEBIDAS QUENTES,TÉRREO,11° CARTÓRIO,...,NaN,QUINTA-FEIRA,X,X,X,X,X,NaN,NaN,5
2,SP,30504.0,908079.0,10653,10653,MAQ VENDING BQ NECTA COLIBRI C4,500,BEBIDAS QUENTES,COPA,CONSELHO TUTELAR - PERUS,...,NaN,TERÇA-FEIRA,X,NaN,X,NaN,X,NaN,NaN,3
3,SP,19571.0,73456.0,A16962,A16962,MAQ VENDING BQ COFFEEMAX III STD GER. II,120,BEBIDAS QUENTES,1 ANDAR,2ºCARTORIO,...,NaN,TERÇA-FEIRA,X,X,X,X,X,NaN,NaN,5
4,SP,35477.0,35614.0,028666,28666,MAQ VENDING BQ NW KREA ONE TOUCH,220,BEBIDAS QUENTES,10 ANDAR COPA,99 TAXIS,...,NaN,QUARTA-FEIRA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3553,SP,38024.0,25158.0,031829,31829,MAQ VENDING BQ NW KREA ONE TOUCH,220,BEBIDAS QUENTES,NaN,YAMAHA JANDIRA,...,NaN,SEXTA-FEIRA,X,X,X,X,X,NaN,NaN,5
3554,SP,38024.0,25158.0,031831,31831,MAQ VENDING BQ NW KREA ONE TOUCH,220,BEBIDAS QUENTES,NaN,YAMAHA JANDIRA,...,NaN,SEXTA-FEIRA,X,X,X,X,X,NaN,NaN,5
3555,SP,38024.0,25158.0,034756,34756,MAQ VENDING BQ NW KREA ONE TOUCH,220,BEBIDAS QUENTES,NaN,YAMAHA JANDIRA,...,NaN,SEXTA-FEIRA,X,X,X,X,X,NaN,NaN,5
3556,SP,38024.0,25158.0,030905,30905,MAQ VENDING BQ NW KREA ONE TOUCH,220,BEBIDAS QUENTES,COPA,YAMAHA JANDIRA,...,NaN,SEXTA-FEIRA,X,X,X,X,X,NaN,NaN,5


In [7]:

df = df.dropna(subset=[lat_col, lon_col])

# garantir que lat/lon sejam numéricos
df[lat_col] = pd.to_numeric(df[lat_col], errors="coerce")
df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")
df = df.dropna(subset=[lat_col, lon_col])

for sup in supervisores:
    df_sup = df[df[supervisor_col].astype(str).str.strip().str.upper() == sup.upper()].copy()

    if df_sup.empty:
        print(f"Atenção: sem dados para o supervisor '{sup}'. Nenhum mapa será gerado para ele(a).")
        continue

    # centro do mapa: média das coordenadas (pode usar outro ponto)
    center_lat = df_sup[lat_col].mean()
    center_lon = df_sup[lon_col].mean()

    m = folium.Map(location=[center_lat, center_lon], zoom_start=11)

    # cluster para muitos pontos
    cluster = MarkerCluster().add_to(m)

    for _, row in df_sup.iterrows():
        lat = float(row[lat_col])
        lon = float(row[lon_col])
        label = str(row.get(label_col, ""))

        popup_html = folium.Popup(label, parse_html=True)
        folium.Marker(location=[lat, lon], popup=popup_html, tooltip=label).add_to(cluster)

    # salvar
    safe_name = sup.replace(" ", "_")
    out_path = os.path.join(output_folder, f"mapa_{safe_name}.html")
    m.save(out_path)
    print(f"Mapa salvo: {out_path}")

print("Concluído.")


C:\Users\tbekho01.ATKEARNEY_AD\AppData\Local\Temp\ipykernel_24320\801945693.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[lat_col] = pd.to_numeric(df[lat_col], errors="coerce")
C:\Users\tbekho01.ATKEARNEY_AD\AppData\Local\Temp\ipykernel_24320\801945693.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df[lon_col] = pd.to_numeric(df[lon_col], errors="coerce")


Mapa salvo: C:\Users\tbekho01.ATKEARNEY_AD\Kearney\Gran Coffee - Otimização de despesas operacionais - Project Management\5. Working Folder\24. Handover\250826_Otimização de Abastecedores_vEnviado\Dados Intermediários\mapa_ALAN_FONTES.html
Mapa salvo: C:\Users\tbekho01.ATKEARNEY_AD\Kearney\Gran Coffee - Otimização de despesas operacionais - Project Management\5. Working Folder\24. Handover\250826_Otimização de Abastecedores_vEnviado\Dados Intermediários\mapa_ANDRÉ.html
Concluído.
